# openEO backup — Sentinel-2 indices and dNBR

This compact workflow replaces the main cloud-EO steps used in:

- **Practical 01:** Sentinel-2 orientation, NDVI and NBR;
- **Practical 03:** pre/post NBR and dNBR;
- **Practical 05:** pre-fire NDVI and NDMI.

The canonical GEE notebooks remain the primary course material. Use this notebook only when a trainer switches the class to the CDSE openEO backup.

To keep processing fast, the workflow uses a small buffered bounding box around EFFIS target polygon **240575**, not the full course AOI.

### Kernel

Run this fallback in the **same Python 3 kernel used for the canonical GEE notebooks**.

The cell below installs only the lightweight `openeo` Python client when it is missing. Do not install GeoPandas, Rasterio, xarray or other compiled geospatial packages into CDSE's dedicated OpenEO kernel just for this course.

In [ ]:
import importlib.util
import subprocess
import sys

# The openEO client is intentionally the only package installed at runtime.
# The canonical Python 3 course kernel already provides the geospatial stack.
if importlib.util.find_spec("openeo") is None:
    print("Installing the openEO Python client in the current session...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "openeo>=0.50,<1",
    ])

required_existing = ["geopandas", "numpy", "matplotlib", "xarray", "netCDF4"]
missing_existing = [
    name for name in required_existing
    if importlib.util.find_spec(name) is None
]

if missing_existing:
    raise RuntimeError(
        "This fallback should run in the same Python 3 kernel as the GEE "
        "practicals. Missing core package(s): "
        + ", ".join(missing_existing)
        + ". Do not repair this by installing compiled geospatial packages "
        "into the dedicated OpenEO kernel; switch to the course Python 3 kernel."
    )

print("Fallback kernel ready.")

## 1. Imports and course target area

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import openeo
import xarray as xr

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parents[1] if len(Path.cwd().parents) > 1 else Path.cwd(),
        Path.home() / "mystorage" / "fire-school",
    ]
    for candidate in candidates:
        if (candidate / "data/effis/Galicica.gpkg").exists():
            return candidate
    raise FileNotFoundError("Could not find the fire-school repository.")

REPO_ROOT = find_repo_root()
EFFIS_PATH = REPO_ROOT / "data/effis/Galicica.gpkg"

effis = gpd.read_file(EFFIS_PATH).to_crs("EPSG:4326")
target = effis[effis["id"].astype(str) == "240575"].copy()
if target.empty:
    raise RuntimeError("EFFIS target polygon 240575 was not found.")

west, south, east, north = target.total_bounds
# Keep the emergency live request compact. This still leaves useful
# context around EFFIS polygon 240575 while reducing openEO processing time.
BUFFER_DEG = 0.01

BBOX = {
    "west": float(west - BUFFER_DEG),
    "south": float(south - BUFFER_DEG),
    "east": float(east + BUFFER_DEG),
    "north": float(north + BUFFER_DEG),
    "crs": "EPSG:4326",
}

print("Backup processing extent:", BBOX)

## 2. Connect and authenticate

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()
print("openEO authenticated.")

## 3. Reusable Sentinel-2 composite helper

CDSE provides the `to_scl_dilation_mask` process for Sentinel-2 Scene Classification Layer masking. This is slightly different from the simple SCL mask in the canonical GEE notebooks, so exact pixels may differ.

In [ ]:
MAX_CLOUD = 80

def s2_median(start, end, bands):
    scl = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=BBOX,
        temporal_extent=[start, end],
        bands=["SCL"],
        max_cloud_cover=MAX_CLOUD,
    )

    cloud_mask = scl.process(
        "to_scl_dilation_mask",
        data=scl,
        kernel1_size=17,
        kernel2_size=77,
        mask1_values=[2, 4, 5, 6, 7],
        mask2_values=[3, 8, 9, 10, 11],
        erosion_kernel_size=3,
    )

    data = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=BBOX,
        temporal_extent=[start, end],
        bands=bands,
        max_cloud_cover=MAX_CLOUD,
    )

    return data.mask(cloud_mask).reduce_temporal("median")

## 4. Build the pre/post fire composites and indices

In [ ]:
pre = s2_median(
    "2024-06-01",
    "2024-08-05",
    ["B04", "B08", "B11", "B12"],
)

post = s2_median(
    "2024-08-19",
    "2024-10-01",
    ["B04", "B08", "B11", "B12"],
)

prefire_july = s2_median(
    "2024-07-01",
    "2024-08-05",
    ["B04", "B08", "B11"],
)

def normalized_difference(cube, band_a, band_b):
    a = cube.band(band_a)
    b = cube.band(band_b)
    return (a - b) / (a + b)

nbr_pre_raw = normalized_difference(pre, "B08", "B12")
nbr_post_raw = normalized_difference(post, "B08", "B12")

# The openEO Python client cannot do direct band-math between DataCubes
# that originate from different process-graph roots. Merge the two
# one-band NBR cubes first, then subtract inside a band reducer.
nbr_pre = nbr_pre_raw.add_dimension(
    name="bands",
    label="NBR_pre",
    type="bands",
)
nbr_post = nbr_post_raw.add_dimension(
    name="bands",
    label="NBR_post",
    type="bands",
)
nbr_pair = nbr_pre.merge_cubes(nbr_post)

from openeo.processes import array_element

dnbr = nbr_pair.reduce_dimension(
    dimension="bands",
    reducer=lambda data: (
        array_element(data=data, index=0)
        - array_element(data=data, index=1)
    ),
).add_dimension(
    name="bands",
    label="dNBR",
    type="bands",
)

ndvi_prefire_raw = normalized_difference(prefire_july, "B08", "B04")
ndmi_prefire_raw = normalized_difference(prefire_july, "B08", "B11")
ndvi_prefire = ndvi_prefire_raw.add_dimension(
    name="bands",
    label="NDVI_prefire",
    type="bands",
)
ndmi_prefire = ndmi_prefire_raw.add_dimension(
    name="bands",
    label="NDMI_prefire",
    type="bands",
)

bundle = (
    nbr_pre
    .merge_cubes(nbr_post)
    .merge_cubes(dnbr)
    .merge_cubes(ndvi_prefire)
    .merge_cubes(ndmi_prefire)
)

print("Process graph ready. No cloud processing has run yet.")

## 5. Execute the compact target-area request

The first live openEO execution can take several minutes. Results are therefore cached under persistent `~/mystorage/geo_adapt_openeo_cache/` when CDSE persistent storage is available.

Subsequent runs reuse the cached NetCDF file instead of processing the same request again.

In [ ]:
mystorage = Path.home() / "mystorage"
CACHE_DIR = (
    mystorage / "geo_adapt_openeo_cache"
    if mystorage.exists()
    else Path("/tmp/geo_adapt_openeo")
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = CACHE_DIR / "s2_indices_dnbr.nc"

if OUTPUT.exists() and OUTPUT.stat().st_size > 0:
    print("Using cached openEO result:", OUTPUT)
else:
    print(
        "No cached result found. The first live CDSE openEO request can take "
        "several minutes; later runs reuse this file."
    )
    try:
        bundle.download(str(OUTPUT), format="NetCDF")
        print("Synchronous result:", OUTPUT)
    except Exception as exc:
        print("Synchronous request did not finish:", exc)
        print("Switching to a batch job...")
        bundle.execute_batch(
            outputfile=str(OUTPUT),
            out_format="NetCDF",
            title="GEO-ADAPT openEO backup: S2 indices + dNBR",
        )
        print("Batch result:", OUTPUT)

## 6. Inspect the result

In [ ]:
EXPECTED_BANDS = [
    ("NBR_pre", "NBR pre-fire", -1.0, 1.0, "RdYlGn"),
    ("NBR_post", "NBR post-fire", -1.0, 1.0, "RdYlGn"),
    ("dNBR", "dNBR", -2.0, 2.0, "RdYlGn_r"),
    ("NDVI_prefire", "NDVI pre-fire", -1.0, 1.0, "RdYlGn"),
    ("NDMI_prefire", "NDMI pre-fire", -1.0, 1.0, "RdYlGn"),
]

ds = xr.load_dataset(OUTPUT, engine="netcdf4")
print(ds)

def extract_band(dataset, expected_name, fallback_index):
    if expected_name in dataset.data_vars:
        return dataset[expected_name].squeeze(drop=True)

    variables = list(dataset.data_vars)

    # Some openEO NetCDF exports keep all bands in one variable with a
    # band-like dimension instead of one variable per band.
    if len(variables) == 1:
        da = dataset[variables[0]]
        band_dim = next(
            (d for d in da.dims if d.lower() in {"band", "bands"}),
            None,
        )
        if band_dim is not None and da.sizes[band_dim] > fallback_index:
            return da.isel({band_dim: fallback_index}).squeeze(drop=True)

    if fallback_index < len(variables):
        return dataset[variables[fallback_index]].squeeze(drop=True)

    raise RuntimeError(
        f"Could not locate '{expected_name}' in NetCDF variables {variables}."
    )

for i, (expected, title, physical_min, physical_max, cmap) in enumerate(EXPECTED_BANDS):
    da = extract_band(ds, expected, i)
    raw = np.asarray(da.values, dtype="float32")
    raw = np.squeeze(raw)

    finite = np.isfinite(raw)
    physical = finite & (raw >= physical_min) & (raw <= physical_max)
    arr = np.where(physical, raw, np.nan)

    excluded = int(finite.sum() - physical.sum())
    if not np.isfinite(arr).any():
        raise RuntimeError(
            f"No physically plausible finite values remain for {title}."
        )

    vmin = float(np.nanmin(arr))
    vmax = float(np.nanmax(arr))
    median = float(np.nanmedian(arr))

    print(
        f"{title:16s}",
        "min =", round(vmin, 3),
        "median =", round(median, 3),
        "max =", round(vmax, 3),
        f"| excluded out-of-range pixels: {excluded}",
    )

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(
        arr,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
    )
    ax.set_title(f"{title} — stretched to valid min/max")
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.show()

ds.close()

## 7. Interpretation questions

1. Is the 2024 disturbance visible as higher dNBR in a spatially coherent area?
2. Do the pre-fire NDVI and NDMI maps emphasize the same places?
3. Why might this result differ slightly from the GEE notebook even with the same dates?
4. Which differences are caused by backend implementation, and which would change the scientific conclusion?

The backend is secondary. The evidence and its limitations remain the focus.